# AU-Fish: Prepare Fish-Only COCO Dataset

Notebook ini membuat `annotations_fish_only.json` untuk protokol `time_based` dan `source_based` tanpa menyalin gambar. Semua gambar, `image_id`, urutan gambar, dan `file_name` dipertahankan. Gambar tanpa fish tetap menjadi negative samples.

**Struktur utama yang didukung**
```text
dataset_v2/{time_based|source_based}/{train|val|test}/images
dataset_v2/{time_based|source_based}/{train|val|test}/annotations.json
```

Output dibuat berdampingan sebagai `annotations_fish_only.json`; file asli tidak ditimpa.

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## Konversi dan validasi

Path default adalah `/content/drive/MyDrive/au-fish/dataset_v2`. Ubah `DATASET_ROOT` pada bagian konfigurasi di bawah apabila folder Drive berbeda, lalu jalankan cell.

In [2]:
from __future__ import annotations

import copy
import csv
import json
from collections import Counter
from pathlib import Path
from typing import Any


# ============================================================
# KONFIGURASI COLAB
# ============================================================

# Struktur terbaru yang diasumsikan:
#
# dataset_v2/
# ├── time_based/
# │   ├── train/
# │   │   ├── images/
# │   │   └── annotations.json
# │   ├── val/
# │   └── test/
# └── source_based/
#     ├── train/
#     ├── val/
#     └── test/
#
# Output dibuat di samping annotations.json:
# annotations_fish_only.json
#
# Gambar tidak disalin, dipindahkan, atau diubah.

DATASET_ROOT = Path(
    "/content/drive/MyDrive/au-fish/dataset_v2"
)

PROTOCOLS = ("time_based", "source_based")
SPLITS = ("train", "val", "test")

FISH_CLASS_NAME = "fish"
NEW_FISH_CATEGORY_ID = 1
NEW_FISH_CATEGORY_NAME = "fish"
NEW_FISH_SUPERCATEGORY = "animal"

INPUT_ANNOTATION_FILENAME = "annotations.json"
OUTPUT_ANNOTATION_FILENAME = "annotations_fish_only.json"

OVERWRITE_OUTPUT = True

REPORT_CSV = DATASET_ROOT / "fish_only_conversion_report.csv"
REPORT_JSON = DATASET_ROOT / "fish_only_conversion_report.json"


# ============================================================
# PATH DISCOVERY
# ============================================================

def find_annotation_file(
    dataset_root: Path,
    protocol: str,
    split: str,
) -> Path:
    """Mendukung beberapa pola struktur COCO yang umum."""
    candidates = [
        dataset_root / protocol / split / INPUT_ANNOTATION_FILENAME,
        dataset_root / protocol / "annotations" / f"{split}.json",
        dataset_root / protocol / split / f"{split}.json",
        dataset_root / protocol / split / "_annotations.coco.json",
    ]

    for candidate in candidates:
        if candidate.exists() and candidate.is_file():
            return candidate

    tried = "\n".join(f"  - {candidate}" for candidate in candidates)
    raise FileNotFoundError(
        f"Anotasi tidak ditemukan untuk {protocol}/{split}.\n"
        f"Lokasi yang diperiksa:\n{tried}"
    )


def resolve_output_file(input_json: Path) -> Path:
    """Menaruh JSON fish-only di folder split yang sama."""
    return input_json.with_name(OUTPUT_ANNOTATION_FILENAME)


# ============================================================
# VALIDASI COCO INPUT
# ============================================================

def require_list(
    data: dict[str, Any],
    key: str,
    source: Path,
) -> list[Any]:
    value = data.get(key)
    if not isinstance(value, list):
        raise ValueError(f"'{key}' harus berupa list pada {source}")
    return value


def validate_coco_input(
    data: dict[str, Any],
    source: Path,
) -> None:
    """Memeriksa konsistensi dasar COCO sebelum konversi."""
    images = require_list(data, "images", source)
    annotations = require_list(data, "annotations", source)
    categories = require_list(data, "categories", source)

    image_ids = [
        image.get("id")
        for image in images
        if isinstance(image, dict)
    ]
    if len(image_ids) != len(images):
        raise ValueError(f"Ada record image yang bukan object di {source}")
    if len(set(image_ids)) != len(image_ids):
        raise ValueError(f"Image ID duplikat ditemukan di {source}")

    annotation_ids = [
        annotation.get("id")
        for annotation in annotations
        if isinstance(annotation, dict)
    ]
    if len(annotation_ids) != len(annotations):
        raise ValueError(f"Ada record annotation yang bukan object di {source}")
    if len(set(annotation_ids)) != len(annotation_ids):
        raise ValueError(f"Annotation ID duplikat ditemukan di {source}")

    category_ids = [
        category.get("id")
        for category in categories
        if isinstance(category, dict)
    ]
    if len(category_ids) != len(categories):
        raise ValueError(f"Ada record category yang bukan object di {source}")
    if len(set(category_ids)) != len(category_ids):
        raise ValueError(f"Category ID duplikat ditemukan di {source}")

    image_id_set = set(image_ids)
    category_id_set = set(category_ids)

    invalid_image_refs = [
        annotation.get("id")
        for annotation in annotations
        if annotation.get("image_id") not in image_id_set
    ]
    if invalid_image_refs:
        raise ValueError(
            f"Ada {len(invalid_image_refs)} anotasi dengan image_id tidak valid di {source}"
        )

    invalid_category_refs = [
        annotation.get("id")
        for annotation in annotations
        if annotation.get("category_id") not in category_id_set
    ]
    if invalid_category_refs:
        raise ValueError(
            f"Ada {len(invalid_category_refs)} anotasi dengan category_id tidak valid di {source}"
        )


def find_fish_category(
    categories: list[dict[str, Any]],
    source: Path,
) -> dict[str, Any]:
    matches = [
        category
        for category in categories
        if str(category.get("name", "")).strip().casefold()
        == FISH_CLASS_NAME.casefold()
    ]

    if not matches:
        available = [str(category.get("name", "")) for category in categories]
        raise ValueError(
            f"Kategori '{FISH_CLASS_NAME}' tidak ditemukan di {source}. "
            f"Kategori tersedia: {available}"
        )
    if len(matches) > 1:
        raise ValueError(
            f"Lebih dari satu kategori bernama '{FISH_CLASS_NAME}' di {source}"
        )
    return matches[0]


# ============================================================
# KONVERSI
# ============================================================

def atomic_write_json(destination: Path, data: dict[str, Any]) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as file:
        json.dump(data, file, ensure_ascii=False, indent=2)
    temporary.replace(destination)


def process_split(protocol: str, split: str) -> dict[str, Any]:
    input_json = find_annotation_file(DATASET_ROOT, protocol, split)
    output_json = resolve_output_file(input_json)

    if output_json.exists() and not OVERWRITE_OUTPUT:
        raise FileExistsError(
            f"Output sudah ada dan overwrite dimatikan: {output_json}"
        )

    with input_json.open("r", encoding="utf-8-sig") as file:
        original_data = json.load(file)

    if not isinstance(original_data, dict):
        raise ValueError(f"Root JSON bukan object: {input_json}")

    validate_coco_input(original_data, input_json)

    images = require_list(original_data, "images", input_json)
    annotations = require_list(original_data, "annotations", input_json)
    categories = require_list(original_data, "categories", input_json)

    fish_category = find_fish_category(categories, input_json)
    original_fish_id = fish_category["id"]

    # Salin object penuh agar metadata tambahan tidak hilang.
    fish_only_data = copy.deepcopy(original_data)

    new_annotations: list[dict[str, Any]] = []
    for annotation in annotations:
        if annotation.get("category_id") != original_fish_id:
            continue

        new_annotation = copy.deepcopy(annotation)
        # Annotation ID dan image_id dipertahankan.
        new_annotation["category_id"] = NEW_FISH_CATEGORY_ID
        new_annotations.append(new_annotation)

    fish_only_data["images"] = copy.deepcopy(images)
    fish_only_data["annotations"] = new_annotations
    fish_only_data["categories"] = [
        {
            "id": NEW_FISH_CATEGORY_ID,
            "name": NEW_FISH_CATEGORY_NAME,
            "supercategory": NEW_FISH_SUPERCATEGORY,
        }
    ]

    # Pastikan daftar gambar, image_id, dan file_name identik.
    original_image_ids = [image["id"] for image in images]
    new_image_ids = [image["id"] for image in fish_only_data["images"]]
    if original_image_ids != new_image_ids:
        raise RuntimeError(f"Urutan atau image_id berubah pada {protocol}/{split}")

    original_file_names = [image.get("file_name") for image in images]
    new_file_names = [image.get("file_name") for image in fish_only_data["images"]]
    if original_file_names != new_file_names:
        raise RuntimeError(f"file_name berubah pada {protocol}/{split}")

    validate_coco_input(fish_only_data, output_json)
    atomic_write_json(output_json, fish_only_data)

    all_image_ids = set(original_image_ids)
    original_any_annotation_image_ids = {
        annotation["image_id"] for annotation in annotations
    }
    fish_annotation_image_ids = {
        annotation["image_id"] for annotation in new_annotations
    }

    negative_after = all_image_ids - fish_annotation_image_ids
    newly_negative_nonfish_only = (
        original_any_annotation_image_ids - fish_annotation_image_ids
    )
    already_empty = all_image_ids - original_any_annotation_image_ids

    original_category_counts = Counter(
        annotation["category_id"] for annotation in annotations
    )

    return {
        "protocol": protocol,
        "split": split,
        "input_json": str(input_json),
        "output_json": str(output_json),
        "original_fish_category_id": original_fish_id,
        "new_fish_category_id": NEW_FISH_CATEGORY_ID,
        "original_images": len(images),
        "fish_only_images": len(fish_only_data["images"]),
        "original_annotations": len(annotations),
        "fish_annotations_kept": len(new_annotations),
        "annotations_removed": len(annotations) - len(new_annotations),
        "negative_images_after": len(negative_after),
        "newly_negative_nonfish_only": len(newly_negative_nonfish_only),
        "already_empty_images": len(already_empty),
        "image_ids_identical": original_image_ids == new_image_ids,
        "file_names_identical": original_file_names == new_file_names,
        "original_category_annotation_counts": {
            str(category_id): count
            for category_id, count in sorted(
                original_category_counts.items(),
                key=lambda item: str(item[0]),
            )
        },
        "status": "success",
    }


# ============================================================
# REPORT
# ============================================================

REPORT_FIELDS = [
    "protocol",
    "split",
    "input_json",
    "output_json",
    "original_fish_category_id",
    "new_fish_category_id",
    "original_images",
    "fish_only_images",
    "original_annotations",
    "fish_annotations_kept",
    "annotations_removed",
    "negative_images_after",
    "newly_negative_nonfish_only",
    "already_empty_images",
    "image_ids_identical",
    "file_names_identical",
    "status",
]


def write_reports(results: list[dict[str, Any]]) -> None:
    REPORT_CSV.parent.mkdir(parents=True, exist_ok=True)

    with REPORT_CSV.open("w", encoding="utf-8-sig", newline="") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=REPORT_FIELDS,
            extrasaction="ignore",
        )
        writer.writeheader()
        writer.writerows(results)

    with REPORT_JSON.open("w", encoding="utf-8") as file:
        json.dump(results, file, ensure_ascii=False, indent=2)


def print_report(results: list[dict[str, Any]]) -> None:
    print("\n" + "=" * 128)
    print("LAPORAN VALIDASI DATASET FISH-ONLY")
    print("=" * 128)
    print(
        f"{'Protocol':<14} "
        f"{'Split':<6} "
        f"{'Images':>8} "
        f"{'Fish Ann':>10} "
        f"{'Removed':>10} "
        f"{'Negatives':>11} "
        f"{'New Neg.':>9} "
        f"{'IDs':>5} "
        f"{'Names':>7}"
    )
    print("-" * 128)

    for result in results:
        print(
            f"{result['protocol']:<14} "
            f"{result['split']:<6} "
            f"{result['fish_only_images']:>8} "
            f"{result['fish_annotations_kept']:>10} "
            f"{result['annotations_removed']:>10} "
            f"{result['negative_images_after']:>11} "
            f"{result['newly_negative_nonfish_only']:>9} "
            f"{str(result['image_ids_identical']):>5} "
            f"{str(result['file_names_identical']):>7}"
        )

    print("=" * 128)

    all_valid = all(
        result["original_images"] == result["fish_only_images"]
        and result["image_ids_identical"]
        and result["file_names_identical"]
        for result in results
    )

    if all_valid:
        print(
            "VALID: semua split mempertahankan jumlah gambar, image_id, "
            "urutan gambar, dan file_name yang identik."
        )
        print(
            "Gambar tanpa fish tetap dipertahankan sebagai negative samples."
        )
    else:
        print(
            "PERINGATAN: ditemukan ketidaksesuaian. "
            "Periksa report sebelum training."
        )

    print(f"CSV report : {REPORT_CSV}")
    print(f"JSON report: {REPORT_JSON}")


# ============================================================
# MAIN
# ============================================================

def main() -> None:
    if not DATASET_ROOT.exists():
        raise FileNotFoundError(
            f"DATASET_ROOT tidak ditemukan:\n{DATASET_ROOT}\n\n"
            "Ubah DATASET_ROOT pada cell konfigurasi agar sesuai Google Drive."
        )

    results: list[dict[str, Any]] = []

    for protocol in PROTOCOLS:
        for split in SPLITS:
            print(f"Memproses {protocol}/{split}...")
            result = process_split(protocol, split)
            results.append(result)
            print(f"  Berhasil: {result['output_json']}")

    write_reports(results)
    print_report(results)


if __name__ == "__main__":
    main()


Memproses time_based/train...
  Berhasil: /content/drive/MyDrive/au-fish/dataset_v2/time_based/train/annotations_fish_only.json
Memproses time_based/val...
  Berhasil: /content/drive/MyDrive/au-fish/dataset_v2/time_based/val/annotations_fish_only.json
Memproses time_based/test...
  Berhasil: /content/drive/MyDrive/au-fish/dataset_v2/time_based/test/annotations_fish_only.json
Memproses source_based/train...
  Berhasil: /content/drive/MyDrive/au-fish/dataset_v2/source_based/train/annotations_fish_only.json
Memproses source_based/val...
  Berhasil: /content/drive/MyDrive/au-fish/dataset_v2/source_based/val/annotations_fish_only.json
Memproses source_based/test...
  Berhasil: /content/drive/MyDrive/au-fish/dataset_v2/source_based/test/annotations_fish_only.json

LAPORAN VALIDASI DATASET FISH-ONLY
Protocol       Split    Images   Fish Ann    Removed   Negatives  New Neg.   IDs   Names
-----------------------------------------------------------------------------------------------------------

## Pemeriksaan file output

Cell ini menampilkan semua file fish-only yang dibuat.

In [ ]:
for protocol in PROTOCOLS:
    for split in SPLITS:
        source = find_annotation_file(DATASET_ROOT, protocol, split)
        output = resolve_output_file(source)
        print(f'{protocol}/{split}: {output} | exists={output.exists()}')
